In [54]:
import fiftyone
import fiftyone.utils.random
import pathlib
import yaml
from ultralytics import YOLO


# set up the paths, names and settings

In [55]:
# select to models parent folder
models_parent_dir = pathlib.Path.cwd().parent / "models"
datasets_parent_dir = pathlib.Path.cwd().parent / "datasets"

# choose a name for the new merged dataset
merged_dataset_name = "der_merger"

# Load settings
with open('settings.yaml') as f:
    settings = yaml.safe_load(f)

splits = ["train", "test", "val"]

# setup the datasets in fiftyone

In [ ]:
datasets_to_reset = [merged_dataset_name, "tmp_dataset"]

for dataset in datasets_to_reset:
    try:
        fiftyone.delete_dataset(dataset)
        print(f"Deleted existing dataset: {dataset}")
    except:
        print(f"No existing dataset named: {dataset}")

tmp_dataset = fiftyone.Dataset("tmp_dataset")

# choose a new dataset name to combine the others in
merged_dataset = fiftyone.Dataset(merged_dataset_name)

# choose a dataset to add the predictions to
source_dataset = fiftyone.load_dataset("coco-2017")
tmp_dataset.add_samples(source_dataset)

datasets_to_add = [fiftyone.load_dataset("ebi-fused-4")]

Deleted existing dataset: der_merger
Deleted existing dataset: tmp_dataset
 100% |███████████████| 1500/1500 [13.3s elapsed, 0s remaining, 316.4 samples/s]     


# model inferencing on a particular subset

In [57]:
# choose models to inference with, they must match the settings.yaml
chosen_models = ["doors", "windows", "lights"]

conditions = []

# Apply all models to same field, then filter once
for model_name in chosen_models:
    model = YOLO(models_parent_dir / model_name / "best.pt")
    tmp_dataset.apply_model(model, label_field="predictions")

    for class_name, threshold in settings["models"][model_name].items():
        conditions.append(
            (fiftyone.ViewField("label") == class_name) & 
            (fiftyone.ViewField("confidence") >= threshold)
        )

# Apply combined filter
if conditions:
    keep_condition = conditions[0]
    for condition in conditions[1:]:
        keep_condition |= condition
    
    # Create filtered view
    filtered_view = tmp_dataset.filter_labels("predictions", keep_condition)
    
    # Merge from filtered view
    filtered_view.merge_labels(in_field="predictions", out_field="ground_truth")

# Delete predictions field from the original dataset
# tmp_dataset.delete_sample_fields("predictions")
tmp_dataset.save()

 100% |███████████████| 1500/1500 [15.7s elapsed, 0s remaining, 99.8 samples/s]       
 100% |███████████████| 1500/1500 [16.6s elapsed, 0s remaining, 83.4 samples/s]      
 100% |███████████████| 1500/1500 [17.1s elapsed, 0s remaining, 94.6 samples/s]      


In [58]:
# session = fiftyone.launch_app(tmp_dataset)

# merge datasets

In [59]:
datasets_to_add.append(tmp_dataset)


for dataset in datasets_to_add:
    print(len(dataset))
    merged_dataset.merge_samples(dataset)

merged_dataset.save()
print(merged_dataset)

3398
1500
Name:        der_merger
Media type:  image
Num samples: 4898
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    predictions:      fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)


# redistribute splits

In [60]:

print(f"Splits before redistribution: {merged_dataset.count_sample_tags()}")
merged_dataset.untag_samples(splits)
fiftyone.utils.random.random_split(merged_dataset, {"train": 0.7, "test": 0.2, "val": 0.1})
print(f"Splits after redistribution: {merged_dataset.count_sample_tags()}")

Splits before redistribution: {'val': 2379, 'test': 1179, 'validation': 500, 'train': 840}
Splits after redistribution: {'val': 490, 'test': 979, 'validation': 500, 'train': 3429}


# some class renaming

In [61]:
merged_dataset.set_values(
    "ground_truth.detections.label",
    "Chair",
    fiftyone.ViewField("ground_truth.detections.label") == "chair"
)
merged_dataset.set_values(
    "ground_truth.detections.label",
    "Window",
    fiftyone.ViewField("ground_truth.detections.label") == "window"
)
merged_dataset.set_values(
    "ground_truth.detections.label",
    "Light",
    fiftyone.ViewField("ground_truth.detections.label") == "light"
)

In [62]:


export_dir = datasets_parent_dir / "multiclass" / f"{merged_dataset_name}-{len(merged_dataset)}"

# All splits must use the same classes list
classes = ["Chair", "Window", "Light", "Door"]


# Export the splits
for split in splits:
    split_view = merged_dataset.match_tags(split)
    split_view.export(
        export_dir=str(export_dir),
        dataset_type=fiftyone.types.YOLOv5Dataset,
        label_field="ground_truth",
        # overwrite=True,
        split=split,
        classes=classes,
        progress=True,
    )

Directory '/home/freeze/GIT/onemanstreasure/datasets/multiclass/der_merger-4898' already exists; export will be merged with existing files
 100% |███████████████| 3429/3429 [5.2s elapsed, 0s remaining, 587.0 samples/s]      
Directory '/home/freeze/GIT/onemanstreasure/datasets/multiclass/der_merger-4898' already exists; export will be merged with existing files
 100% |█████████████████| 979/979 [1.5s elapsed, 0s remaining, 319.9 samples/s]         
Directory '/home/freeze/GIT/onemanstreasure/datasets/multiclass/der_merger-4898' already exists; export will be merged with existing files
 100% |█████████████████| 490/490 [749.5ms elapsed, 0s remaining, 653.8 samples/s]      
